# Adversarial Robustness of Fraud Detection Models

Attacking the three fraud-detection models from the [parent project](https://github.com/lakshyakaviya/credit-card-fraud-detection) to measure how easily a fraudster could evade them by manipulating the transaction amount.

**Question:** does predictive accuracy predict adversarial robustness?

---

## Setup

Imports, GPU device, and the feature/attack modules from `src/`.

In [ ]:
import pandas as pd, numpy as np, joblib, torch
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, precision_recall_curve
from sklearn.linear_model import LogisticRegression
import torch.nn as nn
from src.features import fit_feature_pipeline, transform_features, make_Xy
from src.attacks import (
    make_adversarial_amounts, run_amount_attack_xgb, run_amount_attack_sklearn,
    run_amount_attack_torch, threshold_for_recall, security_evaluation_index
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Data and the victim model

Rebuild the time-ordered splits and the leakage-safe feature pipeline from raw data, then load the trained XGBoost model. The PR-AUC check confirms the model and pipeline reproduce the parent project's held-out result exactly before any attack is run.

In [ ]:
raw = pd.read_parquet("data/train_data_merged.parquet").sort_values("TransactionDT").reset_index(drop=True)
n = len(raw)
tr = raw.iloc[:int(n*0.70)].copy()
te = raw.iloc[int(n*0.85):].copy()

state = fit_feature_pipeline(tr)
X_train, y_train = make_Xy(transform_features(tr, state), state)
X_test,  y_test  = make_Xy(transform_features(te, state), state)

xgb = joblib.load("models/xgb_baseline.pkl")
print("XGBoost test PR-AUC:", round(average_precision_score(y_test, xgb.predict_proba(X_test)[:,1]), 4))

## Threat model and attack set

**Manipulable:** `TransactionAmt` only — the one feature a fraudster genuinely controls at transaction time.

**Coupled:** `amt_z_for_card` is *derived* from the amount (its z-score against the card's spending history), so it is recomputed whenever the amount changes rather than perturbed independently.

**Frozen:** the remaining 424 features — card-level aggregates, `V`/`id` blocks, counting and timedelta features, categorical identifiers. A fraudster controls none of these.

The attack set is the frauds the model **currently catches**: evading a detection the model never made would be meaningless.

In [ ]:
THRESHOLD = 0.30
feat_list   = state["feature_cols"]
amt_idx     = feat_list.index("TransactionAmt")
amtz_idx    = feat_list.index("amt_z_for_card")
amtmean_idx = feat_list.index("card_amt_mean")
amtstd_idx  = feat_list.index("card_amt_std")

test_probs  = xgb.predict_proba(X_test)[:, 1]
attack_mask = (y_test.values == 1) & (test_probs >= THRESHOLD)

Xa = X_test[attack_mask].values.astype(np.float64).copy()
orig_amounts = Xa[:, amt_idx].copy()
card_means   = Xa[:, amtmean_idx]
card_stds    = Xa[:, amtstd_idx]
print("Attackable frauds:", Xa.shape[0])

## XGBoost: degradation curve and SEI

Sweep the attack budget ρ from 0 (no change permitted) to 2 (±200% of the original amount). At each budget, a transaction counts as evaded if *any* of 50 candidate amounts drops its fraud score below the decision threshold.

The **Security Evaluation Index (SEI)** is the normalised area under the resulting `Acc(ρ)` curve (Xiao et al., eq. 11–12) — the average share of detections retained across the attack range. Range [0, 1]; higher is more secure.

*Sanity check: ρ = 0 must give 0% evasion.*

In [ ]:
rho_values = [0.0, 0.1, 0.2, 0.3, 0.5, 0.75, 1.0, 1.5, 2.0]
rho_arr = np.array(rho_values)

results = []
for rho in rho_values:
    cand_r = make_adversarial_amounts(orig_amounts, rho=rho, n_steps=50)
    evaded_r, _ = run_amount_attack_xgb(Xa, cand_r, xgb, THRESHOLD,
                                        amt_idx, amtz_idx, card_means, card_stds)
    results.append((rho, evaded_r.mean(), 1 - evaded_r.mean()))
    print(f"rho = {rho:>4} | evaded: {evaded_r.mean()*100:5.1f}% | Acc: {(1-evaded_r.mean())*100:5.1f}%")

sei_xgb = security_evaluation_index(rho_arr, [r[2] for r in results])
print(f"\nXGBoost SEI: {sei_xgb:.4f}")

## Logistic regression: preprocessing and model

Unlike XGBoost, the linear and neural models cannot handle missing values and require scaled inputs. Median imputation and standardisation are both fit on **training data only** and applied forward, mirroring how these models were trained.

In [ ]:
train_medians = X_train.median()
median_vals   = train_medians.values.astype(np.float64)
X_train_f     = X_train.fillna(train_medians).astype(np.float32)
mu    = X_train_f.mean(axis=0)
sigma = X_train_f.std(axis=0).replace(0, 1.0)
X_train_sc = ((X_train_f - mu) / sigma).values.astype(np.float32)

X_test_f  = X_test.fillna(train_medians).astype(np.float32)
X_test_sc = ((X_test_f - mu) / sigma).values.astype(np.float32)

lr = LogisticRegression(max_iter=5000, class_weight="balanced")
lr.fit(X_train_sc, y_train)
lr_test_probs = lr.predict_proba(X_test_sc)[:, 1]
print("LR converged in", lr.n_iter_[0], "iterations")

## Neural network: architecture and weights

Reconstruct the MLP architecture and load the weights trained in the parent project. Scoring runs on GPU.

In [ ]:
class FraudMLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),        nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1),
        )
    def forward(self, x):
        return self.net(x)

model = FraudMLP(X_test_sc.shape[1]).to(device)
model.load_state_dict(torch.load("models/fraud_mlp.pt", map_location=device, weights_only=True))
model.eval()

with torch.no_grad():
    mlp_test_probs = torch.sigmoid(
        model(torch.tensor(X_test_sc, dtype=torch.float32, device=device))
    ).cpu().numpy().ravel()
print("MLP loaded.")

## Recall-matched operating points

Raw probability thresholds are **not comparable across model families** — a score of 0.5 means something different for XGBoost, logistic regression, and a neural net, because each calibrates probabilities differently. Comparing security at a shared raw threshold conflates genuine robustness with the accident of where each default cutoff sits: a stricter threshold *inflates* apparent robustness, since only the model's highest-confidence (hardest to flip) predictions are ever attacked.

Each model's threshold is therefore tuned so that **all three catch the same 82.2% of fraud on clean data** before the attack begins. All three then start from an equivalent detection system, and any difference in degradation is attributable to the models themselves.

In [ ]:
target_recall = ((test_probs >= THRESHOLD) & (y_test.values == 1)).sum() / (y_test.values == 1).sum()
lr_thr  = threshold_for_recall(lr_test_probs,  y_test.values, target_recall)
mlp_thr = threshold_for_recall(mlp_test_probs, y_test.values, target_recall)
print(f"Target recall: {target_recall:.3f}")
print(f"LR threshold:  {lr_thr:.3f}  |  MLP threshold: {mlp_thr:.3f}")

## Attacking LR and the MLP at matched thresholds

Identical attack, identical budgets, equal starting recall — the fair comparison.

In [ ]:
# LR
lr_mask = (y_test.values == 1) & (lr_test_probs >= lr_thr)
Xa_lr = X_test[lr_mask].values.astype(np.float64).copy()
lr_results = []
for rho in rho_values:
    cand_r = make_adversarial_amounts(Xa_lr[:, amt_idx], rho=rho, n_steps=50)
    evaded_r, _ = run_amount_attack_sklearn(Xa_lr, cand_r, lr, lr_thr,
                                            amt_idx, amtz_idx,
                                            Xa_lr[:, amtmean_idx], Xa_lr[:, amtstd_idx],
                                            median_vals, mu.values, sigma.values)
    lr_results.append((rho, evaded_r.mean(), 1 - evaded_r.mean()))
sei_lr = security_evaluation_index(rho_arr, [r[2] for r in lr_results])

# MLP
mlp_mask = (y_test.values == 1) & (mlp_test_probs >= mlp_thr)
Xa_mlp = X_test[mlp_mask].values.astype(np.float64).copy()
mlp_results = []
for rho in rho_values:
    cand_r = make_adversarial_amounts(Xa_mlp[:, amt_idx], rho=rho, n_steps=50)
    evaded_r, _ = run_amount_attack_torch(Xa_mlp, cand_r, model, mlp_thr,
                                          amt_idx, amtz_idx,
                                          Xa_mlp[:, amtmean_idx], Xa_mlp[:, amtstd_idx],
                                          median_vals, mu.values, sigma.values, device)
    mlp_results.append((rho, evaded_r.mean(), 1 - evaded_r.mean()))
sei_mlp = security_evaluation_index(rho_arr, [r[2] for r in mlp_results])

print("=== Recall-matched SEIs ===")
print(f"XGBoost: {sei_xgb:.4f}   (v1: 0.9067)")
print(f"LR:      {sei_lr:.4f}   (v1: 0.9487)")
print(f"MLP:     {sei_mlp:.4f}   (v1: 0.9668)")
print(f"\nAttack sizes — XGB: {Xa.shape[0]}, LR: {Xa_lr.shape[0]}, MLP: {Xa_mlp.shape[0]}")

# Results: accuracy vs. security

All three curves start together at 100% (recall-matched, no perturbation) and fan out as the budget grows.

**The finding:** the accuracy ranking and the robustness ranking disagree completely. The best classifier (XGBoost) is middling on security; the worst classifier (LR) is the most robust; the middle classifier (MLP) is the least robust. There is no monotonic relationship between predictive performance and adversarial robustness.

XGBoost's characteristic **elbow-then-plateau** reflects its discrete, threshold-based splits: perturbations that cross a split boundary cause sudden score changes, but beyond ρ ≈ 1.0 a hard core of frauds remains un-evadable at *any* budget — the model convicts them on the frozen features the attacker cannot touch.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
for res, name, sei, color in [
    (results,     "XGBoost", sei_xgb, "crimson"),
    (lr_results,  "Logistic Regression", sei_lr, "steelblue"),
    (mlp_results, "Neural Net (MLP)", sei_mlp, "seagreen"),
]:
    ax.plot([r[0] for r in res], [r[2]*100 for r in res], marker="o", color=color,
            label=f"{name} (SEI={sei:.3f})")
ax.set_xlabel("Attack budget ρ (fraction of amount manipulable)")
ax.set_ylabel("Accuracy on attacked frauds (%)")
ax.set_title("Security Evaluation Curves — amount-manipulation attack (recall-matched)")
ax.set_ylim(80, 101); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("images/security_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## Save results

In [ ]:
joblib.dump({
    "results_xgb": results, "results_lr": lr_results, "results_mlp": mlp_results,
    "sei": {"xgb": sei_xgb, "lr": sei_lr, "mlp": sei_mlp},
    "matched_thresholds": {"xgb": THRESHOLD, "lr": lr_thr, "mlp": mlp_thr},
    "target_recall": target_recall,
}, "models/project2_results_corrected.pkl")